In [11]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_train_full_clean.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_test_future.parquet')
test_pred = pd.read_csv(PROJECT_ROOT.parent / 'data' / 'result_v1.csv')

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [12]:
# ============================================================
# 1-3. 피쳐링 결과: 컬럼군 정의
# ============================================================

# 종속변수군
target_col = 'target_maxrise_6h'

# 독립변수군
feature_cols = [
    col for col in train_full.columns
    if col not in ('timestamp', target_col)
]

In [13]:
# ============================================================
# 3-1. 강우사상별 데이터 분할 -> event_id 컬럼 생성
# ============================================================

# 전체 timestamp 추출(중복없이), 시간순 정렬
event_time = (
    train_full[['timestamp']] 
    .drop_duplicates() 
    .sort_values('timestamp') 
    .reset_index(drop = True) 
)

# 차분: diff 계산
event_time['event_time_diff'] = (
    event_time['timestamp']
    .diff()
)

# 차분값이 1시간이 아닌 경우를 event 경계로 추출
cond_split_time = (
    event_time['event_time_diff'] != pd.Timedelta(hours=1)
)

# 강우사상 시간대별 event_id 부여
event_time['event_id'] = cond_split_time.cumsum()

# 시험처가 제시한 강우사상 개수와 맞는지 확인: 122개
assert event_time['event_id'].nunique() == 122, (
    f"주최측이 제시한 강우사상 122개, 현재 {event_time['event_id'].nunique()}개, 오류"
)

# train_full에 event_id 컬럼 붙이기(매핑)
rule_event_id_map = event_time.set_index('timestamp')['event_id']  # 매핑규칙(조인대상 인덱스 필수)

train_full['event_id'] = train_full['timestamp'].map(rule_event_id_map)

assert not train_full['event_id'].isnull().any(), (
    'event_id 매핑 누락 발생'
)

## 4. Tidal, Normal 데이터셋 분할

In [14]:
# ============================================================
# Tidal / Normal 데이터 분리
# ============================================================

# tidal 관측소 9개 (기존과 동일)
tidal_stations = [
    'ST0110', 'ST0326', 'ST0159',
    'ST0358', 'ST0347', 'ST0415',
    'ST0292', 'ST0313', 'ST0234'
]

# 4-1. Train 분리
tidal_train = train_full.loc[
    train_full['station_id'].isin(tidal_stations)
].copy()

normal_train = train_full.loc[
    ~train_full['station_id'].isin(tidal_stations)
].copy()

# 4-2. Test 분리
tidal_test = test.loc[
    test['station_id'].isin(tidal_stations)
].copy()

normal_test = test.loc[
    ~test['station_id'].isin(tidal_stations)
].copy()

# 4-3. 분리 결과 확인
print('<Train 분리 결과>')
print(f'Train 전체 : {len(train_full):,}')
print(f'Tidal      : {len(tidal_train):,}')
print(f'Normal     : {len(normal_train):,}')

print('\n<Test 분리 결과>')
print(f'Test 전체  : {len(test):,}')
print(f'Tidal      : {len(tidal_test):,}')
print(f'Normal     : {len(normal_test):,}')

# 4-4. 데이터 분리 검증
assert len(tidal_train) + len(normal_train) == len(train_full)
assert len(tidal_test) + len(normal_test) == len(test)

print('\n데이터 분리 검증 완료')

<Train 분리 결과>
Train 전체 : 749,609
Tidal      : 14,251
Normal     : 735,358

<Test 분리 결과>
Test 전체  : 114,111
Tidal      : 2,169
Normal     : 111,942

데이터 분리 검증 완료


## 5. 독립변수 / 종속변수 분리

In [15]:
# ============================================================
# 5. 독립변수 / 종속변수 분리
# ============================================================

assert len(feature_cols) == 32, (
    f'독립변수 개수 이상: 예상 32개, 현재 {len(feature_cols)}개'
)

# Tidal
X_tidal_train = tidal_train[feature_cols].copy()
X_tidal_test = tidal_test[feature_cols].copy()

y_tidal_train = tidal_train[target_col]

# Normal
X_normal_train = normal_train[feature_cols].copy()
X_normal_test = normal_test[feature_cols].copy()

y_normal_train = normal_train[target_col]

# 데이터 분리 검증
assert X_tidal_train.index.equals(y_tidal_train.index)
assert X_normal_train.index.equals(y_normal_train.index)

assert X_tidal_train.columns.equals(X_tidal_test.columns)
assert X_normal_train.columns.equals(X_normal_test.columns)

print('독립변수 / 종속변수 분리 검증 완료')

독립변수 / 종속변수 분리 검증 완료


## 6. Tidal 데이터셋 모델 생성: 단일 홀드아웃, CatBoost, ExtraTree 앙상블

In [16]:
# ============================================================
# 6-1. CatBoost 범주형 변수 처리
# ============================================================

for X in [
    X_tidal_train, X_tidal_test,
    X_normal_train, X_normal_test
]:
    X['station_id'] = X['station_id'].astype(str)

In [20]:
# ============================================================
# 6-2. CatBoost 모델로 tidal 예측
# ============================================================
from catboost import CatBoostRegressor

# Tidal CatBoost 모델

cat_tidal = CatBoostRegressor()

cat_tidal.load_model(
    str(PROJECT_ROOT.parent / 'model' / 'cat_tidal.cbm')
)

pred_cat_tidal = cat_tidal.predict(X_tidal_test)

In [23]:
# ============================================================
# 6-3. ExtraTrees 모델용 OneHotEncoding
# ============================================================

from sklearn.preprocessing import OneHotEncoder

import joblib

# 저장된 OneHotEncoder 불러오기
ohe = joblib.load(
    PROJECT_ROOT.parent / 'model' / 'ohe_tidal_station_id.joblib'
)

ohe.fit(X_tidal_train[['station_id']])

# Train / Test 원핫인코딩
X_tidal_train_ohe = pd.concat([
    X_tidal_train.drop(columns='station_id'),  # 기존에 있던 'station_id' 컬럼은 제거
    pd.DataFrame(  # 원핫인코딩 결과 데이터프레임으로 만들어서 붙이기
        ohe.transform(X_tidal_train[['station_id']]),  
        columns=ohe.get_feature_names_out(),  # 컬럼 이름을 해당 station_id로 받아오기
        index=X_tidal_train.index
    )
], axis=1)

X_tidal_test_ohe = pd.concat([
    X_tidal_test.drop(columns='station_id'),
    pd.DataFrame(
        ohe.transform(X_tidal_test[['station_id']]),
        columns=ohe.get_feature_names_out(),
        index=X_tidal_test.index
    )
], axis=1)

# 검증
assert 'station_id' not in X_tidal_train_ohe.columns
assert X_tidal_train_ohe.columns.equals(X_tidal_test_ohe.columns)
assert X_tidal_train_ohe.index.equals(y_tidal_train.index)

# 원핫인코딩 전후 Shape 비교
print(f'Tidal Train : {X_tidal_train.shape} -> {X_tidal_train_ohe.shape}')
print(f'Tidal Test  : {X_tidal_test.shape} -> {X_tidal_test_ohe.shape}')

# 학습된 OneHotEncoder 저장
import joblib

Tidal Train : (14251, 32) -> (14251, 40)
Tidal Test  : (2169, 32) -> (2169, 40)


In [24]:
# ============================================================
# 6-4. ExtraTrees 모델로 tidal 예측
# ============================================================
from sklearn.ensemble import ExtraTreesRegressor

# Tidal ExtraTrees 모델
et_tidal = joblib.load(
    PROJECT_ROOT.parent / 'model' / 'et_tidal_ExtraTrees.joblib'
)


et_tidal.fit(
    X_tidal_train_ohe, y_tidal_train,
)

pred_et_tidal = et_tidal.predict(X_tidal_test_ohe)

# Tidal ExtraTrees 모델 저장
import joblib

In [25]:
# ============================================================
# 6-5. CatBoost 45% + ExtraTrees 55% 앙상블 예측
# ============================================================

pred_ensemble_tidal = pd.Series(
    pred_cat_tidal * 0.45 + pred_et_tidal * 0.55,
    index=X_tidal_test.index
)

# 앙상블 예측 결과 검증
assert pred_ensemble_tidal.index.equals(X_tidal_test.index)
assert len(pred_ensemble_tidal) == len(tidal_test)
assert pred_ensemble_tidal.notna().all()

print(f'Tidal Test: {len(tidal_test):,}행')
print(f'앙상블 예측: {len(pred_ensemble_tidal):,}행')
print('Tidal 앙상블 예측 검증 완료')

Tidal Test: 2,169행
앙상블 예측: 2,169행
Tidal 앙상블 예측 검증 완료


## 7. Normal 데이터셋 모델 생성: K_Fold 교차검증, CatBoost

In [ ]:
# ============================================================
# 7-1. Normal CatBoost 5-Fold 배정 csv 불러와서 매칭
# ============================================================

# 고정 Fold 매핑 파일 불러오기
fold_match = pd.read_csv(
    PROJECT_ROOT / 'data' / '0_kf_train_full_match.csv'
)

# event_id 기준으로 Fold 번호 매칭
normal_train['fold_num'] = normal_train['event_id'].map(
    fold_match.set_index('event_id')['fold_num']  # 매핑규칙: 'event_id'로 조인, 'fold_num' 부여
)

# 매칭 검증
assert normal_train['fold_num'].notna().all()
assert normal_train['fold_num'].between(1, N_SPLITS).all()

print('Normal Fold 매칭 완료')

In [ ]:
# ============================================================
# 7-2. Normal CatBoost 5-Fold 학습 및 예측
# ============================================================

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error
import numpy as np

# 결과 저장용 변수
normal_oof_pred = pd.Series(
    index=X_normal_train.index,
    dtype=float
)

normal_test_preds = pd.DataFrame(
    index=X_normal_test.index
)

normal_fold_results = []

# Fold별 학습
for fold in range(1, N_SPLITS + 1):

    print(f'\n========== Normal Fold {fold} ==========')

    # 기존 Fold 번호로 분할
    train_mask = normal_train['fold_num'] != fold
    valid_mask = normal_train['fold_num'] == fold

    X_kf_tr = X_normal_train.loc[train_mask]
    X_kf_val = X_normal_train.loc[valid_mask]

    y_kf_tr = y_normal_train.loc[train_mask]
    y_kf_val = y_normal_train.loc[valid_mask]

    assert len(X_kf_tr) > 0 and len(X_kf_val) > 0, f'Fold {fold}: 학습/검증 데이터 없음'

    # CatBoost 최종 선정모델
    model = CatBoostRegressor(
        iterations=1100,  # train 5-Fold에서 fold별 best iteration 편차가 큼 -> 평균 선택
        random_seed=42,
        cat_features=['station_id'],
        thread_count=-1
    )

    model.fit(
        X_kf_tr, y_kf_tr,
        verbose=500
    )

    # Fold별 kf_val 예측
    val_pred = model.predict(X_kf_val)

    # Fold별 OOF 예측 및 RMSE
    normal_oof_pred.loc[X_kf_val.index] = val_pred
    fold_oof_rmse = root_mean_squared_error(y_kf_val, val_pred)

    # Fold별 Test 전체 예측
    test_pred = model.predict(X_normal_test)
    normal_test_preds[f'fold_{fold}'] = test_pred  # Fold별 예측결과가 하나의 컬럼으로 삽입
  
    # Fold 결과 저장
    normal_fold_results.append({
        'fold': fold,
        'train_count': len(X_kf_tr),
        'valid_count': len(X_kf_val),
        'oof_rmse': fold_oof_rmse,
    })

    print(
        f'Fold {fold} | '
        f'OOF RMSE={fold_oof_rmse:.6f} | '
    )

    # Normal CatBoost 모델 저장
    model.save_model(
        str(PROJECT_ROOT / 'model' / f"cat_{fold}fold_normal.cbm")
    )

In [ ]:
# ============================================================
# 7-2. Normal CatBoost 학습 및 예측 결과 검증
# ============================================================

# OOF 예측 검증
assert normal_oof_pred.index.equals(y_normal_train.index)
assert normal_oof_pred.notna().all()

# Test 예측 검증
assert normal_test_preds.index.equals(X_normal_test.index)
assert normal_test_preds.shape == (len(X_normal_test), N_SPLITS)
assert normal_test_preds.notna().all().all()

# 전체 OOF RMSE
normal_oof_rmse = root_mean_squared_error(
    y_normal_train, normal_oof_pred
)

print(f'Normal 전체 OOF RMSE: {normal_oof_rmse:.6f}')
print(f'Normal Test 예측: {normal_test_preds.shape}')
print('Normal CatBoost 검증 완료')

In [ ]:
# ============================================================
# 7-3. Normal CatBoost 5-Fold 앙상블 예측
# ============================================================

pred_ensemble_normal = normal_test_preds.mean(axis=1)  # axis=1은 행별집계
# 결과물: 하나의 행에 fold별 예측값이 컬럼별로 배치 -> 행별집계 -> 행별 1개 mean값(인덱스 유지)

# 검증
assert pred_ensemble_normal.index.equals(X_normal_test.index)
assert pred_ensemble_normal.notna().all()

print(f'Normal Test: {len(normal_test):,}행')
print(f'앙상블 예측: {len(pred_ensemble_normal):,}행')
print('Normal 앙상블 예측 검증 완료')

## 8. Tidal / Normal 예측 결과 통합 및 제출파일 생성

In [ ]:
# ============================================================
# 8. Tidal / Normal 예측 결과 통합 및 제출파일 생성
# ============================================================

# 예측 결과 통합
pred_test = pd.concat([
    pred_ensemble_tidal,
    pred_ensemble_normal
])

# 제출용 데이터프레임 생성
final_result = pred_test.rename('pred_maxrise_6h').reset_index()

# 검증
assert final_result.columns.tolist() == ['row_id', 'pred_maxrise_6h']
assert len(final_result) == len(test)
assert final_result['row_id'].is_unique
assert set(final_result['row_id']) == set(test.index)
assert final_result['pred_maxrise_6h'].notna().all()

print('제출 데이터 생성 및 검증 완료')
print(final_result.shape)

In [ ]:
final_result.to_csv(
    PROJECT_ROOT / 'data' / 'result_v1.csv',
    index=False,
    encoding='utf-8'
)

In [ ]:
final_result.head(100)